First preparing data for training FFNN

In [10]:
import pandas as pd
import numpy as np
from model_functions.FFNN import FFNN
from sklearn.preprocessing import StandardScaler

train= pd.read_csv('claims_train.csv')
test = pd.read_csv('claims_test.csv')

#Cleaning weird values that found during cleaning
train = train[train['Exposure'] <= 1].copy()
test =test[test['Exposure']<=1].copy()

y_train_np = train["ClaimNb"].to_numpy().reshape(-1, 1)
y_test_np  = test["ClaimNb"].to_numpy().reshape(-1, 1)
y_test=test['ClaimNb']
# Adding Risk and dropping correlated and useless columns
train=train.drop(columns=['IDpol', 'ClaimNb','Exposure'])
test=test.drop(columns=['IDpol', 'ClaimNb','Exposure'])

#Encoding
train_encoded=pd.get_dummies(train, columns=['VehBrand', 'VehGas', 'Region'], drop_first=True)#Encoding categorical values
area_map={'A':1,'B':2,'C':3,'D':4,'E':5,'F':6}
train_encoded['Area']=train_encoded['Area'].map(area_map)

test_encoded=pd.get_dummies(test, columns=['VehBrand', 'VehGas', 'Region'], drop_first=True)#Encoding categorical values
area_map={'A':1,'B':2,'C':3,'D':4,'E':5,'F':6}
test_encoded['Area']=test_encoded['Area'].map(area_map)

#Standarizing values
Scaler = StandardScaler()
train_encoded = Scaler.fit_transform(train_encoded)
test_encoded  = Scaler.transform(test_encoded)

X_train_np = np.asarray(train_encoded, dtype=float)
X_test_np  = np.asarray(test_encoded, dtype=float)




Time to use the self-made FFNN model. We decided it would makes sense to choose 64 neurons for first hidden layer and 32 for the second one

In [12]:

model=FFNN(input_size=38,hidden_sizes=[64,32],output_size=1, lr=0.0005)
model.fit(X=X_train_np,y=y_train_np,epochs=10, batch_size=1024)
y_pred=model.predict(X=X_test_np)

import numpy as np
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    mean_poisson_deviance,
    confusion_matrix
)

poisson_dev = mean_poisson_deviance(y_test_np, y_pred)
print(f"Poisson Deviance: {poisson_dev:.4f}")

#Binary Classification
y_true_bin = (y_test_np > 0).astype(int)
#Threshold for classification
threshold = 0.1
y_pred_bin = (y_pred > threshold).astype(int)

# Confusion matrix
cm = confusion_matrix(y_true_bin, y_pred_bin)
print("\nConfusion Matrix:")
print(cm)


# Precision, Recall, F1
precision = precision_score(y_true_bin, y_pred_bin, zero_division=0)
recall = recall_score(y_true_bin, y_pred_bin, zero_division=0)
f1 = f1_score(y_true_bin, y_pred_bin, zero_division=0)

print(f"\nPrecision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-score:  {f1:.4f}")

y_pred_flat = np.array(y_pred).reshape(-1)
df=pd.DataFrame({"y_test":y_test, "y_pred":y_pred_flat})
df["bucket"] = pd.qcut(df.y_pred, q=10, labels=False)

df.groupby("bucket")["y_test"].mean()





Epoch 0, Loss=0.3506974458928151
Epoch 1, Loss=0.28387181246621307
Epoch 2, Loss=0.22982886531006658
Epoch 3, Loss=0.23377666150054263
Epoch 4, Loss=0.21103487219374872
Epoch 5, Loss=0.21122008273968
Epoch 6, Loss=0.2303277247218792
Epoch 7, Loss=0.23637359156268697
Epoch 8, Loss=0.18349465754837535
Epoch 9, Loss=0.2379793459111563
Poisson Deviance: 0.3841

Confusion Matrix:
[[87880 40654]
 [ 4727  2112]]

Precision: 0.0494
Recall:    0.3088
F1-score:  0.0852


bucket
0    0.052667
1    0.052006
2    0.054591
3    0.055625
4    0.060792
5    0.054296
6    0.050513
7    0.054533
8    0.051636
9    0.050894
Name: y_test, dtype: float64